# CS801 Part 2 - IEEE-CIS Fraud Detection: quantitative EDA and validation

This notebook investigates fraud-detection data using quantitative exploratory analysis and validation design. It combines formal tests, effect sizes, class-imbalance-aware metrics, a time-aware validation split, robust handling of missing identity columns, and a lightweight baseline model that shows how EDA findings can inform modelling without leaking information from the test set.

In [ ]:
# MW-METHOD #12 - Imports and project configuration
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

from scipy.stats import chi2_contingency, ks_2samp
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_recall_curve
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.append(str(PROJECT_ROOT))

from src.cs801_utils import (
    safe_read_csv, existing_columns, cramers_v, benjamini_hochberg,
    mannwhitney_effect, binary_classification_summary,
)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 120)

In [ ]:
# MW-METHOD #13 - Data loading
DATA_DIR = PROJECT_ROOT / "data" / "raw"
train_transaction = safe_read_csv(DATA_DIR / "fraud_train_transaction.csv")
train_identity = safe_read_csv(DATA_DIR / "fraud_train_identity.csv")

# Test files are loaded only for optional unlabeled distribution checks, not for model selection.
test_transaction_path = DATA_DIR / "fraud_test_transaction.csv"
test_identity_path = DATA_DIR / "fraud_test_identity.csv"
test_transaction = safe_read_csv(test_transaction_path) if test_transaction_path.exists() else None
test_identity = safe_read_csv(test_identity_path) if test_identity_path.exists() else None

print("train_transaction", train_transaction.shape)
print("train_identity", train_identity.shape)
if test_transaction is not None:
    print("test_transaction", test_transaction.shape)
if test_identity is not None:
    print("test_identity", test_identity.shape)

In [ ]:
# MW-METHOD #14 - Safe merge and identity availability feature
identity_overlap = train_transaction["TransactionID"].isin(train_identity["TransactionID"]).mean()
print(f"Share of training transactions with identity data: {identity_overlap:.2%}")

train = train_transaction.merge(train_identity, on="TransactionID", how="left", validate="one_to_one")
train["has_identity"] = train["TransactionID"].isin(train_identity["TransactionID"]).astype(int)

# Work in chronological order because TransactionDT reveals that this competition is time-split.
train = train.sort_values("TransactionDT").reset_index(drop=True)
display(train[["TransactionID", "TransactionDT", "TransactionAmt", "ProductCD", "isFraud", "has_identity"]].head())

In [ ]:
# MW-METHOD #15 - Class imbalance quantified with appropriate metric implications
class_counts = train["isFraud"].value_counts().sort_index().rename(index={0: "not_fraud", 1: "fraud"})
fraud_rate = train["isFraud"].mean()
display(pd.DataFrame({"count": class_counts, "proportion": class_counts / len(train)}))
print(f"Fraud prevalence: {fraud_rate:.3%}")
print("Implication: accuracy alone is inappropriate; use recall, precision, F1, ROC-AUC and especially PR-AUC/average precision.")

plt.figure(figsize=(6, 3))
sns.countplot(data=train, x="isFraud")
plt.title("Class distribution: severe rare-event imbalance")
plt.show()

In [ ]:
# MW-METHOD #16 - Time-aware train/validation split for EDA-to-model link
split_index = int(len(train) * 0.80)
eda_train = train.iloc[:split_index].copy()
eda_valid = train.iloc[split_index:].copy()
print("EDA/training period:", eda_train["TransactionDT"].min(), "to", eda_train["TransactionDT"].max(), eda_train.shape)
print("Validation period:", eda_valid["TransactionDT"].min(), "to", eda_valid["TransactionDT"].max(), eda_valid.shape)
print("Validation is later in time, which better reflects the competition setup than random splitting.")

In [ ]:
# MW-METHOD #17 - Formal numerical tests: fraud vs non-fraud distributions
# Mann-Whitney U is used because many transaction features are skewed and non-normal.
MAX_PER_CLASS = 50_000
num_candidates = ["TransactionAmt", "dist1", "dist2"] + [f"C{i}" for i in range(1, 15)] + [f"D{i}" for i in range(1, 16)]
num_candidates = existing_columns(eda_train, num_candidates)

num_test_rows = []
for col in num_candidates:
    fraud = eda_train.loc[eda_train["isFraud"] == 1, col].dropna()
    nonfraud = eda_train.loc[eda_train["isFraud"] == 0, col].dropna()
    if len(fraud) >= 20 and len(nonfraud) >= 20:
        fraud_sample = fraud.sample(min(len(fraud), MAX_PER_CLASS), random_state=RANDOM_STATE)
        nonfraud_sample = nonfraud.sample(min(len(nonfraud), MAX_PER_CLASS), random_state=RANDOM_STATE)
        result = mannwhitney_effect(fraud_sample, nonfraud_sample)
        result.update({
            "feature": col,
            "fraud_median": fraud.median(),
            "nonfraud_median": nonfraud.median(),
            "fraud_missing_rate": fraud.isna().mean() if len(fraud) else np.nan,
        })
        num_test_rows.append(result)

num_tests = pd.DataFrame(num_test_rows)
if not num_tests.empty:
    num_tests["p_adj_bh"] = benjamini_hochberg(num_tests["p_value"])
    display(num_tests.sort_values(["p_adj_bh", "rank_biserial"], ascending=[True, False]).head(25))

In [ ]:
# MW-METHOD #18 - Formal categorical tests: conditional fraud rates and effect sizes
cat_candidates = ["ProductCD", "card4", "card6", "P_emaildomain", "R_emaildomain", "DeviceType"] + [f"M{i}" for i in range(1, 10)]
cat_candidates = existing_columns(eda_train, cat_candidates)

cat_test_rows = []
for col in cat_candidates:
    tmp = eda_train[[col, "isFraud"]].copy()
    tmp[col] = tmp[col].astype("object").fillna("<missing>")
    # Collapse extremely rare levels to make chi-square expected counts more stable.
    counts = tmp[col].value_counts()
    rare_levels = counts[counts < 50].index
    tmp[col] = tmp[col].where(~tmp[col].isin(rare_levels), "<rare>")
    table = pd.crosstab(tmp[col], tmp["isFraud"])
    if table.shape[0] > 1 and table.shape[1] == 2:
        chi2, p_value, dof, expected = chi2_contingency(table)
        rates = tmp.groupby(col)["isFraud"].agg(["size", "mean"]).sort_values("mean", ascending=False).head(3)
        cat_test_rows.append({
            "feature": col,
            "chi2": chi2,
            "p_value": p_value,
            "cramers_v": cramers_v(table),
            "top_fraud_rate_levels": rates.to_dict("index"),
        })

cat_tests = pd.DataFrame(cat_test_rows)
if not cat_tests.empty:
    cat_tests["p_adj_bh"] = benjamini_hochberg(cat_tests["p_value"])
    display(cat_tests.sort_values(["p_adj_bh", "cramers_v"], ascending=[True, False]).head(20))

In [ ]:
# MW-METHOD #19 - Transaction amount: log transform plus formal test
plot_df = eda_train[["TransactionAmt", "isFraud"]].dropna().copy()
plot_df["log_TransactionAmt"] = np.log1p(plot_df["TransactionAmt"])

plt.figure(figsize=(8, 4))
sns.kdeplot(data=plot_df.sample(min(len(plot_df), 80_000), random_state=RANDOM_STATE), x="log_TransactionAmt", hue="isFraud", common_norm=False)
plt.title("Log transaction amount by fraud label")
plt.show()

tx_amt_test = mannwhitney_effect(
    plot_df.loc[plot_df["isFraud"] == 1, "log_TransactionAmt"].sample(min((plot_df["isFraud"] == 1).sum(), MAX_PER_CLASS), random_state=RANDOM_STATE),
    plot_df.loc[plot_df["isFraud"] == 0, "log_TransactionAmt"].sample(min((plot_df["isFraud"] == 0).sum(), MAX_PER_CLASS), random_state=RANDOM_STATE),
)
display(pd.DataFrame([tx_amt_test]))

In [ ]:
# MW-METHOD #20 - High-dimensional V-features: PCA instead of one simple row-wise mean
v_cols = [c for c in eda_train.columns if c.startswith("V")]
if v_cols:
    v_sample = eda_train[v_cols].sample(min(len(eda_train), 40_000), random_state=RANDOM_STATE)
    v_sample = v_sample.replace([np.inf, -np.inf], np.nan).fillna(v_sample.median(numeric_only=True))
    v_sample = v_sample.fillna(0)
    v_pca = PCA(n_components=min(20, len(v_cols)), random_state=RANDOM_STATE)
    v_pca.fit(v_sample)
    v_report = pd.DataFrame({
        "component": np.arange(1, len(v_pca.explained_variance_ratio_) + 1),
        "explained_variance_ratio": v_pca.explained_variance_ratio_,
        "cumulative_variance": np.cumsum(v_pca.explained_variance_ratio_),
    })
    display(v_report.head(20))
    plt.figure(figsize=(8, 4))
    plt.plot(v_report["component"], v_report["cumulative_variance"], marker="o")
    plt.title("PCA structure within V-features")
    plt.xlabel("V-feature principal components")
    plt.ylabel("Cumulative explained variance")
    plt.show()
else:
    print("No V-features found.")

In [ ]:
# MW-METHOD #21 - Baseline model that respects time split and class imbalance
# This is intentionally lightweight: the goal is to connect EDA to evaluation, not to win the competition.
base_numeric = ["TransactionAmt", "dist1", "dist2", "card1", "card2", "card3", "card5"] + [f"C{i}" for i in range(1, 15)] + [f"D{i}" for i in range(1, 16)]
base_categorical = ["ProductCD", "card4", "card6", "P_emaildomain", "R_emaildomain", "DeviceType", "has_identity"] + [f"M{i}" for i in range(1, 10)]
base_numeric = existing_columns(train, base_numeric)
base_categorical = existing_columns(train, base_categorical)
feature_cols = base_numeric + base_categorical

X_train = eda_train[feature_cols]
y_train = eda_train["isFraud"]
X_valid = eda_valid[feature_cols]
y_valid = eda_valid["isFraud"]

numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
    ("scaler", StandardScaler()),
])
categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=50)),
])
preprocess = ColumnTransformer(transformers=[
    ("num", numeric_pipeline, base_numeric),
    ("cat", categorical_pipeline, base_categorical),
])

baseline_model = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", LogisticRegression(max_iter=1000, class_weight="balanced", n_jobs=-1, solver="saga", C=0.5)),
])

baseline_model.fit(X_train, y_train)
valid_scores = baseline_model.predict_proba(X_valid)[:, 1]
summary_05 = binary_classification_summary(y_valid, valid_scores, threshold=0.5)
display(pd.DataFrame([summary_05]).T.rename(columns={0: "value"}))

In [ ]:
# MW-METHOD #22 - Threshold tuning using precision-recall trade-off
precision, recall, thresholds = precision_recall_curve(y_valid, valid_scores)
f1_values = 2 * precision * recall / np.maximum(precision + recall, 1e-12)
# thresholds has length len(precision)-1, so skip the last precision/recall point.
best_idx = int(np.nanargmax(f1_values[:-1])) if len(thresholds) else 0
best_threshold = float(thresholds[best_idx]) if len(thresholds) else 0.5
summary_best = binary_classification_summary(y_valid, valid_scores, threshold=best_threshold)

threshold_report = pd.DataFrame([summary_05, summary_best], index=["threshold_0.50", "threshold_max_f1"])
display(threshold_report)

plt.figure(figsize=(7, 4))
plt.plot(recall, precision)
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-recall curve on later validation period")
plt.show()

In [ ]:
# MW-METHOD #23 - Optional unlabeled covariate-shift check, separated from modelling
# This section does not use test labels and should not drive supervised model selection.
if test_transaction is not None:
    shift_features = [c for c in ["TransactionAmt", "card1", "addr1", "dist1"] if c in train_transaction.columns and c in test_transaction.columns]
    shift_rows = []
    for col in shift_features:
        train_sample = pd.to_numeric(train_transaction[col], errors="coerce").dropna().sample(min(50_000, train_transaction[col].notna().sum()), random_state=RANDOM_STATE)
        test_sample = pd.to_numeric(test_transaction[col], errors="coerce").dropna().sample(min(50_000, test_transaction[col].notna().sum()), random_state=RANDOM_STATE)
        if len(train_sample) > 10 and len(test_sample) > 10:
            stat, p_value = ks_2samp(train_sample, test_sample)
            shift_rows.append({"feature": col, "ks_statistic": stat, "p_value": p_value})
    shift_report = pd.DataFrame(shift_rows)
    if not shift_report.empty:
        shift_report["p_adj_bh"] = benjamini_hochberg(shift_report["p_value"])
        display(shift_report.sort_values("ks_statistic", ascending=False))
else:
    print("Test transaction file not available; skipping optional covariate-shift check.")

## Conclusion for Part 2

The analysis treats fraud detection as a rare-event, time-structured classification problem. Categorical features are assessed with chi-square tests and Cramer's V, numeric features with Mann-Whitney tests and rank-biserial effects, high-dimensional V-features with PCA, and evaluation uses precision, recall, F1, ROC-AUC and PR-AUC instead of accuracy. The baseline model is deliberately simple but time-aware, demonstrating how EDA can inform fraud modelling while avoiding test-label leakage.